# Advanced RAG — SEC Filings

**Pipeline**: Metadata Extraction → Query Rewriting → Multi-Query → Hybrid Search (BM25 + Dense + RRF) → CrossEncoder Reranking → Context Compression → LLM Generation with Citations

## What's new vs Baseline RAG?

| Stage | Baseline | Advanced |
|-------|----------|----------|
| Query prep | Raw query | Rewrite + 2 variants |
| Retrieval | Dense only | BM25 + Dense + RRF |
| Scope | All chunks | Metadata-filtered |
| Scoring | Cosine sim | CrossEncoder rerank |
| Context | Full chunks | Sentence-level compression |
| Answer | Plain text | Text with [n] citations |

Each improvement is independently motivated:
1. **Metadata filtering** — restricts search to the relevant company/year, reducing irrelevant noise
2. **Query rewriting** — aligns query vocabulary with financial document language
3. **Multi-query** — different phrasings capture different relevant chunks
4. **Hybrid BM25 + Dense** — BM25 handles exact financial term matching; dense handles semantic similarity
5. **CrossEncoder reranking** — more accurate relevance scoring than bi-encoder cosine similarity
6. **Context compression** — removes off-topic sentences, reduces LLM context noise
7. **Citation generation** — improves traceability and faithfulness

In [1]:
print('hi')

hi


In [2]:
# Uncomment to install dependencies
# !pip install sentence-transformers chromadb rank-bm25 groq python-dotenv pandas tqdm

## 1. Configuration

In [3]:
CONFIG = {
    # --- Paths (relative to this notebook) ---
    "chroma_db_path": "../sec_rag_team_share/chroma_db",
    "chunks_path":    "../sec_rag_team_share/sec_data/chunks/sec_chunks.jsonl",
    "eval_path":      "../sec_rag_team_share/evaluation/GenAI Eval QA.csv",
    "results_dir":    "./results",

    # --- Models (same as langgraph_agentic_rag_sec_v2) ---
    "dense_model_name":   "sentence-transformers/all-mpnet-base-v2",
    "reranker_model_name":"cross-encoder/ms-marco-MiniLM-L-6-v2",
    # Execution profile: 'dev' or 'final'
    "execution_profile":  "dev",
    "groq_dev_model":    "llama-3.1-8b-instant",
    "groq_final_model":  "meta-llama/llama-4-scout-17b-16e-instruct",
    "agent_model":       "llama-3.1-8b-instant",  # always fast model for query ops
    "judge_model":       "llama-3.1-8b-instant",

    # --- Retrieval ---
    "bm25_top_k":        8,
    "dense_top_k":       8,
    "rerank_top_k":      5,
    "multi_query_n":     2,   # number of additional query variants to generate
    "rrf_k":             60,  # RRF smoothing constant

    # --- Context compression ---
    "compress_top_sentences": 4,  # sentences to keep per chunk; None to disable

    # --- Generation ---
    "max_context_chars":     9000,
    "temperature_rewriter":  0.2,
    "temperature_generator": 0.2,
    "temperature_judge":     0.0,
    "max_tokens_answer":     512,
    "max_tokens_judge":      256,

    # --- Evaluation ---
    "eval_split": "dev",

    # --- Rate limiting ---
    "max_rpm":                    28,
    "inter_question_sleep_sec":   2.2,
    "llm_max_retries":            3,
    "llm_retry_base_delay_sec":   5,
}

## 2. Imports & Setup

In [4]:
import os
import re
import json
import time
import threading
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from groq import Groq

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "Set GROQ_API_KEY in your .env file"

LLM_MODEL = (
    CONFIG["groq_final_model"]
    if CONFIG["execution_profile"] == "final"
    else CONFIG["groq_dev_model"]
)
print(f"Execution profile : {CONFIG['execution_profile']}")
print(f"LLM model         : {LLM_MODEL}")

c:\Users\jolen\anaconda3\envs\genai_proj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Execution profile : dev
LLM model         : llama-3.1-8b-instant


## 3. Load Models

In [5]:
print("Loading embedding model...")
embed_model = SentenceTransformer(CONFIG["dense_model_name"])
print(f"  ok {CONFIG['dense_model_name']}")

print("Loading reranker...")
reranker = CrossEncoder(CONFIG["reranker_model_name"])
print(f"  ok {CONFIG['reranker_model_name']}")

groq_client = Groq(api_key=GROQ_API_KEY)
print("  ok Groq client ready")


class RateLimiter:
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self._ts: deque = deque()
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()
            while self._ts and now - self._ts[0] >= 60.0:
                self._ts.popleft()
            if len(self._ts) >= self.max_rpm:
                sleep_for = 60.0 - (now - self._ts[0])
                if sleep_for > 0:
                    print(f"  [RateLimit] sleeping {sleep_for:.1f}s ...")
                    time.sleep(sleep_for)
            self._ts.append(time.time())


rate_limiter = RateLimiter(CONFIG["max_rpm"])


def call_llm(
    messages: list,
    model: str = None,
    temperature: float = 0.2,
    max_tokens: int = 512,
    json_mode: bool = False,
) -> str:
    model = model or LLM_MODEL
    kwargs = dict(model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            resp = groq_client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
            print(f"  [WARN] attempt {attempt+1} failed ({e}). Retrying in {delay}s...")
            time.sleep(delay)
    raise RuntimeError("LLM call failed after max retries")

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9986.80it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ok sentence-transformers/all-mpnet-base-v2
Loading reranker...


c:\Users\jolen\anaconda3\envs\genai_proj\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jolen\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8036.24it/s]
BertForSequenceClassif

  ok cross-encoder/ms-marco-MiniLM-L-6-v2
  ok Groq client ready


## 4. Load Data

In [6]:
# --- ChromaDB ---
print("Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(path=CONFIG["chroma_db_path"])
collections = chroma_client.list_collections()
collection = chroma_client.get_collection(collections[0].name)
print(f"  ok '{collections[0].name}'  ({collection.count():,} chunks)")

# --- JSONL chunks (needed for BM25) ---
print("Loading SEC chunks from JSONL...")
raw_chunks = []
with open(CONFIG["chunks_path"], "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_chunks.append(json.loads(line))

chunks_df = pd.DataFrame(raw_chunks)

# Filter low-value chunks (matching the existing pipeline)
if "flag_low_value_combined" in chunks_df.columns:
    before = len(chunks_df)
    chunks_df = chunks_df[~chunks_df["flag_low_value_combined"].fillna(False)].reset_index(drop=True)
    print(f"  Filtered {before - len(chunks_df)} low-value chunks")

print(f"  ok {len(chunks_df):,} chunks loaded")


def make_contextual_chunk(row) -> str:
    """Reconstruct the contextual chunk text (must match what was stored in ChromaDB)."""
    company = row.get("company_name", row.get("company", "?"))
    ticker  = row.get("ticker", "?")
    form    = row.get("form_type", "?")
    date    = str(row.get("filing_date", "?"))[:10]
    year    = row.get("filing_year", "?")
    section = row.get("section_title", "?")
    text    = row.get("text", row.get("raw_chunk", ""))
    return (
        f"Company: {company} ({ticker})\n"
        f"Filing: {form} | Date: {date} | Year: {year}\n"
        f"Section: {section}\n"
        f"Content: {text}"
    )


# Build contextual chunks column and BM25 tokens
chunks_df["contextual_chunk"] = chunks_df.apply(make_contextual_chunk, axis=1)
chunks_df["bm25_tokens"] = chunks_df["contextual_chunk"].str.lower().str.split()

# Build BM25 index
print("Building BM25 index...")
bm25_index = BM25Okapi(chunks_df["bm25_tokens"].tolist())
print(f"  ok BM25 index built  ({len(chunks_df):,} documents)")

Connecting to ChromaDB...
  ok 'sec_filings'  (12,606 chunks)
Loading SEC chunks from JSONL...
  Filtered 546 low-value chunks
  ok 12,606 chunks loaded
Building BM25 index...
  ok BM25 index built  (12,606 documents)


## 5. Step 1 — Metadata Extraction

Extract company ticker, filing year, and form type from the query to narrow retrieval scope.
This reduces the search space from ~12,600 chunks to only the relevant company/period.

In [7]:
# Known tickers in the dataset
KNOWN_TICKERS = {
    "AAPL", "MSFT", "NVDA", "JPM", "BAC", "BRK.B", "BRK",
    "JNJ", "UNH", "XOM", "CVX", "WMT", "COST",
}

# Company name -> ticker mapping for natural language mentions
COMPANY_TO_TICKER = {
    "apple": "AAPL", "microsoft": "MSFT", "nvidia": "NVDA",
    "jpmorgan": "JPM", "jp morgan": "JPM", "bank of america": "BAC",
    "berkshire": "BRK.B", "johnson": "JNJ", "unitedhealth": "UNH",
    "exxon": "XOM", "exxonmobil": "XOM", "chevron": "CVX",
    "walmart": "WMT", "costco": "COST",
}


def extract_metadata(query: str) -> dict:
    """
    Extract ticker, filing_year, and form_type from query text.
    Uses regex for year/form_type and a lookup table for company names.
    Returns dict with None values for fields not found.
    """
    q_upper = query.upper()
    q_lower = query.lower()

    # --- Ticker ---
    ticker = None
    for t in KNOWN_TICKERS:
        if re.search(r"\b" + re.escape(t) + r"\b", q_upper):
            ticker = t
            break
    if ticker is None:
        for name, t in COMPANY_TO_TICKER.items():
            if name in q_lower:
                ticker = t
                break

    # --- Filing year (2020-2029) ---
    year_match = re.search(r"\b(202[0-9])\b", query)
    filing_year = int(year_match.group(1)) if year_match else None

    # --- Form type ---
    form_type = None
    if re.search(r"\b10-K\b|\bannual report\b|\bannual filing\b", query, re.IGNORECASE):
        form_type = "10-K"
    elif re.search(r"\b10-Q\b|\bquarterly report\b|\bquarterly filing\b", query, re.IGNORECASE):
        form_type = "10-Q"

    return {"ticker": ticker, "filing_year": filing_year, "form_type": form_type}


# Quick test
print(extract_metadata("What was Apple's revenue in fiscal 2024 10-K?"))
print(extract_metadata("Compare NVDA and MSFT operating income for 2023"))

{'ticker': 'AAPL', 'filing_year': 2024, 'form_type': '10-K'}
{'ticker': 'MSFT', 'filing_year': 2023, 'form_type': None}


## 6. Step 2 — Query Rewriting

Rewrite the user's question into a retrieval-optimised query.
Financial questions often use colloquial phrasing ("how much did they make") that doesn't match
document language ("net revenue", "operating income"). A single LLM call fixes the vocabulary mismatch.

In [8]:
REWRITE_SYSTEM = (
    "You are a search query optimizer for SEC filings (10-K, 10-Q). "
    "Rewrite the user's question as a concise retrieval query that maximizes recall of relevant "
    "financial document chunks. Use precise financial terminology (e.g., 'net revenue', "
    "'operating income', 'diluted EPS'). Keep it under 25 words. "
    "Return only the rewritten query, nothing else."
)


def rewrite_query(query: str) -> str:
    """Rewrite query for better retrieval alignment with financial document language."""
    return call_llm(
        messages=[
            {"role": "system", "content": REWRITE_SYSTEM},
            {"role": "user",   "content": query},
        ],
        model=CONFIG["agent_model"],
        temperature=CONFIG["temperature_rewriter"],
        max_tokens=80,
    )

## 7. Step 3 — Multi-Query Generation

Generate additional query variants to increase retrieval coverage.
Different phrasings surface different relevant chunks — e.g., asking about "revenue growth"
and "year-over-year revenue change" may retrieve complementary passages.

In [9]:
MULTI_QUERY_SYSTEM = (
    "Generate {n} alternative retrieval queries for the following question about SEC filings. "
    "Each variant should use different financial terminology or focus on a different aspect "
    "of the question to maximise retrieval coverage. "
    'Return a JSON object with key "queries" containing an array of {n} query strings.'
)


def generate_multi_queries(query: str, n: int = None) -> list:
    """Generate n alternative query variants for multi-query retrieval."""
    n = n or CONFIG["multi_query_n"]
    raw = call_llm(
        messages=[
            {"role": "system", "content": MULTI_QUERY_SYSTEM.format(n=n)},
            {"role": "user",   "content": query},
        ],
        model=CONFIG["agent_model"],
        temperature=CONFIG["temperature_rewriter"],
        max_tokens=200,
        json_mode=True,
    )
    try:
        data = json.loads(raw)
        return data.get("queries", [])[:n]
    except Exception:
        return []

## 8. Step 4 — Hybrid Search (BM25 + Dense + RRF)

BM25 excels at exact financial term matching ("EBITDA", "diluted EPS", specific dollar figures).
Dense retrieval excels at semantic similarity.
Reciprocal Rank Fusion (RRF) combines both ranked lists without requiring score normalisation.
Metadata filtering further restricts results to the relevant company/year.

In [16]:
def build_bm25_mask(ticker=None, filing_year=None, form_type=None) -> np.ndarray:
    """Boolean mask to restrict BM25 search to matching metadata."""
    mask = np.ones(len(chunks_df), dtype=bool)
    if ticker:
        mask &= (chunks_df["ticker"].str.upper() == ticker.upper()).values
    if filing_year:
        mask &= (chunks_df["filing_year"].astype(int) == int(filing_year)).values
    if form_type:
        mask &= (chunks_df["form_type"].str.upper() == form_type.upper()).values
    return mask


def bm25_search(query: str, top_k: int, mask: np.ndarray = None) -> list:
    """BM25 retrieval over contextual chunks with optional metadata mask."""
    tokens = query.lower().split()
    scores = bm25_index.get_scores(tokens)
    if mask is not None:
        scores = scores * mask.astype(float)
    top_idx = np.argsort(scores)[::-1]
    # Keep only positive-scoring results
    top_idx = [i for i in top_idx if scores[i] > 0][:top_k]
    results = []
    for i in top_idx:
        row = chunks_df.iloc[i]
        results.append({
            "text":     row["contextual_chunk"],
            "metadata": {
                "ticker":      row.get("ticker", "?"),
                "form_type":   row.get("form_type", "?"),
                "filing_year": int(row.get("filing_year", 0)),
                "company":     row.get("company_name", row.get("company", "?")),
            },
            "score":    float(scores[i]),
            "chunk_id": str(row.get("chunk_id", i)),
        })
    return results


# def dense_search(query: str, top_k: int, ticker=None, filing_year=None, form_type=None) -> list:
#     """Dense (semantic) retrieval with optional ChromaDB metadata filtering."""
#     query_vec = embed_model.encode(query, normalize_embeddings=True).tolist()
#     where = {}
#     if ticker:       where["ticker"]       = {"$eq": ticker}
#     if filing_year:  where["filing_year"]  = {"$eq": int(filing_year)}
#     if form_type:    where["form_type"]    = {"$eq": form_type}
#     kwargs = dict(
#         query_embeddings=[query_vec],
#         n_results=min(top_k, collection.count()),
#         include=["documents", "metadatas", "distances", "ids"],
#     )
#     if where:
#         kwargs["where"] = where
#     results = collection.query(**kwargs)
#     chunks = []
#     for doc, meta, dist, cid in zip(
#         results["documents"][0],
#         results["metadatas"][0],
#         results["distances"][0],
#         results["ids"][0],
#     ):
#         chunks.append({
#             "text":     doc,
#             "metadata": meta,
#             "score":    float(1.0 - dist),
#             "chunk_id": cid,
#         })
#     return chunks

def dense_search(query: str, top_k: int, ticker=None, filing_year=None, form_type=None) -> list:
    """Dense (semantic) retrieval with optional ChromaDB metadata filtering."""
    query_vec = embed_model.encode(query, normalize_embeddings=True).tolist()

    filters = []
    if ticker:
        filters.append({"ticker": {"$eq": ticker}})
    if filing_year:
        filters.append({"filing_year": {"$eq": int(filing_year)}})
    if form_type:
        filters.append({"form_type": {"$eq": form_type}})

    kwargs = dict(
        query_embeddings=[query_vec],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],  # removed "ids"
    )

    if len(filters) == 1:
        kwargs["where"] = filters[0]
    elif len(filters) > 1:
        kwargs["where"] = {"$and": filters}

    results = collection.query(**kwargs)

    chunks = []
    for doc, meta, dist, cid in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
        results["ids"][0],   # ids are still returned automatically
    ):
        chunks.append({
            "text": doc,
            "metadata": meta,
            "score": float(1.0 - dist),
            "chunk_id": cid,
        })

    return chunks


def rrf_merge(result_lists: list, k: int = None) -> list:
    """
    Reciprocal Rank Fusion over multiple ranked lists.
    Formula: score(d) = sum_i 1 / (k + rank_i(d) + 1)
    """
    k = k or CONFIG["rrf_k"]
    scores = {}
    pool   = {}
    for ranked_list in result_lists:
        for rank, chunk in enumerate(ranked_list):
            cid = chunk["chunk_id"]
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
            if cid not in pool:
                pool[cid] = chunk
    merged = []
    for cid in sorted(scores, key=scores.__getitem__, reverse=True):
        c = dict(pool[cid])
        c["score"] = scores[cid]
        merged.append(c)
    return merged

## 9. Step 5 — CrossEncoder Reranking

The bi-encoder embedding model scores query and document independently (fast but approximate).
The CrossEncoder attends to both simultaneously, giving more accurate relevance scores.
Applied to the top RRF candidates to select the final context.

In [11]:
def rerank(candidates: list, query: str, top_k: int = None) -> list:
    """Rerank candidate chunks using CrossEncoder (query, chunk) scoring."""
    top_k = top_k or CONFIG["rerank_top_k"]
    if not candidates:
        return []
    pairs  = [(query, c["text"]) for c in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

## 10. Step 6 — Context Compression

Each retrieved chunk may contain irrelevant sentences.
We extract only the most relevant sentences (scored by CrossEncoder against the query),
reducing noise in the LLM context window.

In [12]:
def compress_chunk(chunk: dict, query: str, top_n: int = None) -> dict:
    """
    Extract the most relevant sentences from a chunk using CrossEncoder scoring.
    Keeps the structured header (Company/Filing/Section) and replaces the content
    with only the top-N most relevant sentences.
    """
    top_n = top_n or CONFIG["compress_top_sentences"]
    if top_n is None:
        return chunk  # compression disabled

    text = chunk["text"]

    # Separate header (Company/Filing/Section lines) from content
    if "\nContent:" in text:
        header, _, content = text.partition("\nContent:")
        header = header + "\nContent:"
    else:
        header = ""
        content = text

    # Simple sentence splitting (handles ., !, ?, ;)
    sentences = re.split(r"(?<=[.!?;])\s+", content.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 25]

    if len(sentences) <= top_n:
        return chunk  # already short enough

    # Score each sentence against the query
    pairs  = [(query, s) for s in sentences]
    scores = reranker.predict(pairs, show_progress_bar=False)

    # Keep top-N sentences in original document order
    top_idx = sorted(
        sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]
    )
    compressed_content = " ".join(sentences[i] for i in top_idx)

    new_chunk = dict(chunk)
    new_chunk["text"] = (header + " " + compressed_content).strip()
    new_chunk["compressed"] = True
    return new_chunk


def compress_context(chunks: list, query: str) -> list:
    """Compress all chunks in the context."""
    return [compress_chunk(c, query) for c in chunks]

## 11. Step 7 — Generation with Citations

Instructs the LLM to cite source chunks using [n] notation.
This improves traceability and allows users to verify claims against the original filings.

In [13]:
GENERATOR_SYSTEM_ADV = (
    "You are a financial analyst answering questions using only the retrieved filing context.\n"
    "Rules:\n"
    "- Be precise with numbers -- include units (millions, billions, %), fiscal periods, and line item names.\n"
    "- Cite your sources using [n] notation after each claim (e.g., 'Revenue was $394B in FY2024 [1]').\n"
    "- Only use information present in the provided context.\n"
    "- If the context does not contain enough information, say 'Insufficient information in the provided filings.'"
)


def format_context_adv(chunks: list, max_chars: int = None) -> str:
    """Format chunks with numbered headers for citation."""
    max_chars = max_chars or CONFIG["max_context_chars"]
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        header = (
            f"[{i}] {m.get('ticker','?')} {m.get('form_type','?')} "
            f"{m.get('filing_year','?')} | {m.get('company','')}"
        )
        parts.append(f"{header}\n{c['text']}")
    context = "\n\n---\n\n".join(parts)
    return context[:max_chars]


def generate_with_citations(query: str, chunks: list) -> str:
    """Generate an answer with [n] citations referencing the retrieved chunks."""
    context  = format_context_adv(chunks)
    user_msg = f"Question:\n{query}\n\nRetrieved context (cite [n] for each claim):\n{context}"
    return call_llm(
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM_ADV},
            {"role": "user",   "content": user_msg},
        ],
        temperature=CONFIG["temperature_generator"],
        max_tokens=CONFIG["max_tokens_answer"],
    )

## 12. Full Advanced RAG Pipeline

In [30]:
def advanced_rag(query: str, verbose: bool = False) -> dict:
    """
    Advanced RAG pipeline:
    1. Metadata extraction  -> filter to relevant company/year
    2. Query rewriting      -> align with financial document vocabulary
    3. Multi-query          -> generate additional query variants
    4. Hybrid retrieval     -> BM25 + Dense for each query variant
    5. RRF merge            -> combine all ranked lists
    6. CrossEncoder rerank  -> accurate relevance re-scoring
    7. Context compression  -> keep only most relevant sentences
    8. Generation           -> answer with [n] citations
    """
    # 1. Metadata extraction
    meta = extract_metadata(query)
    if verbose:
        print(f"  [Meta]     {meta}")

    # 2. Query rewriting
    rewritten = rewrite_query(query)
    if verbose:
        print(f"  [Rewrite]  {rewritten}")

    # 3. Multi-query generation
    variants  = generate_multi_queries(rewritten)
    all_queries = [rewritten] + variants
    if verbose:
        print(f"  [Queries]  {all_queries}")

    # 4. Hybrid retrieval (BM25 + Dense) for each query variant
    bm25_mask = build_bm25_mask(**meta)
    # all_bm25  = []
    # all_dense = []
    # for q in all_queries:
    #     all_bm25.extend(bm25_search(q, CONFIG["bm25_top_k"], bm25_mask))
    #     all_dense.extend(dense_search(q, CONFIG["dense_top_k"], **meta))

    # if verbose:
    #     print(f"  [Hybrid]   {len(all_bm25)} BM25 + {len(all_dense)} dense candidates")

    # # 5. RRF merge
    # candidates = rrf_merge([all_bm25, all_dense])

    ranked_lists = []
    for q in all_queries:
        ranked_lists.append(bm25_search(q, CONFIG["bm25_top_k"], bm25_mask))
        ranked_lists.append(dense_search(q, CONFIG["dense_top_k"], **meta))

    candidates = rrf_merge(ranked_lists)


    if verbose:
        print(f"  [RRF]      {len(candidates)} unique candidates after merge")

    # 6. CrossEncoder reranking
    reranked = rerank(candidates, rewritten)
    if verbose:
        print(f"  [Rerank]   top {len(reranked)} chunks selected")

    # 7. Context compression
    compressed = compress_context(reranked, rewritten)

    # 8. Generation with citations
    context = format_context_adv(compressed)
    answer  = generate_with_citations(query, compressed)

    return {
        "query":      query,
        "rewritten":  rewritten,
        "variants":   variants,
        "metadata":   meta,
        "answer":     answer,
        "chunks":     reranked,
        "context":    context,
        "n_chunks":   len(reranked),
    }

### Quick Demo

In [31]:
demo_q = "What was Apple's total net revenue for fiscal year 2024?"
result = advanced_rag(demo_q, verbose=True)

print()
print("QUESTION:", result["query"])
print()
print("ANSWER:", result["answer"])
print()
print(f"Metadata extracted : {result['metadata']}")
print(f"Rewritten query    : {result['rewritten']}")
print(f"Query variants     : {result['variants']}")
print()
print(f"Retrieved {result['n_chunks']} chunks:")
for i, c in enumerate(result["chunks"], 1):
    m = c["metadata"]
    score = c.get("rerank_score", c.get("score", 0))
    compressed = "(compressed)" if c.get("compressed") else ""
    print(f"  [{i}] {m.get('ticker','?'):6s}  {m.get('form_type','?'):5s}  "
          f"year={m.get('filing_year','?')}  rerank={score:.3f}  {compressed}")

  [Meta]     {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
  [Rewrite]  "Apple 10-K 2024 net revenue fiscal year"
  [Queries]  ['"Apple 10-K 2024 net revenue fiscal year"', 'Apple annual report 2024 net sales fiscal year', 'Apple 10-K 2024 revenue by fiscal year']
  [RRF]      23 unique candidates after merge
  [Rerank]   top 5 chunks selected

QUESTION: What was Apple's total net revenue for fiscal year 2024?

ANSWER: Based on the provided context, the total net revenue for fiscal year 2024 is $391,035 million [4].

Metadata extracted : {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
Rewritten query    : "Apple 10-K 2024 net revenue fiscal year"
Query variants     : ['Apple annual report 2024 net sales fiscal year', 'Apple 10-K 2024 revenue by fiscal year']

Retrieved 5 chunks:
  [1] AAPL    10-Q   year=2024  rerank=7.301  
  [2] AAPL    10-Q   year=2024  rerank=5.640  
  [3] AAPL    10-K   year=2024  rerank=5.418  
  [4] AAPL    10-K   year=2024  rerank=5.288 

In [17]:
demo_q = "What was Apple's total net revenue for fiscal year 2024?"
result = advanced_rag(demo_q, verbose=True)

print()
print("QUESTION:", result["query"])
print()
print("ANSWER:", result["answer"])
print()
print(f"Metadata extracted : {result['metadata']}")
print(f"Rewritten query    : {result['rewritten']}")
print(f"Query variants     : {result['variants']}")
print()
print(f"Retrieved {result['n_chunks']} chunks:")
for i, c in enumerate(result["chunks"], 1):
    m = c["metadata"]
    score = c.get("rerank_score", c.get("score", 0))
    compressed = "(compressed)" if c.get("compressed") else ""
    print(f"  [{i}] {m.get('ticker','?'):6s}  {m.get('form_type','?'):5s}  "
          f"year={m.get('filing_year','?')}  rerank={score:.3f}  {compressed}")

  [Meta]     {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
  [Rewrite]  "Apple 10-K 2024 net revenue fiscal year."
  [Queries]  ['"Apple 10-K 2024 net revenue fiscal year."', 'Apple annual report 2024 net sales fiscal year', 'Apple 10-K 2024 total revenue by fiscal year']
  [Hybrid]   24 BM25 + 24 dense candidates
  [RRF]      23 unique candidates after merge
  [Rerank]   top 5 chunks selected

QUESTION: What was Apple's total net revenue for fiscal year 2024?

ANSWER: Based on the provided context, Apple's total net revenue for fiscal year 2024 is $391,035 million [4].

Metadata extracted : {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
Rewritten query    : "Apple 10-K 2024 net revenue fiscal year."
Query variants     : ['Apple annual report 2024 net sales fiscal year', 'Apple 10-K 2024 total revenue by fiscal year']

Retrieved 5 chunks:
  [1] AAPL    10-Q   year=2024  rerank=7.137  
  [2] AAPL    10-Q   year=2024  rerank=5.289  
  [3] AAPL    10-K   year=2024

In [26]:
demo_q = "What was Apple's total net revenue for fiscal year 2024?"
result = advanced_rag(demo_q, verbose=True)

print()
print("QUESTION:", result["query"])
print()
print("ANSWER:", result["answer"])
print()
print(f"Metadata extracted : {result['metadata']}")
print(f"Rewritten query    : {result['rewritten']}")
print(f"Query variants     : {result['variants']}")
print()
print(f"Retrieved {result['n_chunks']} chunks:")
for i, c in enumerate(result["chunks"], 1):
    m = c["metadata"]
    score = c.get("rerank_score", c.get("score", 0))
    compressed = "(compressed)" if c.get("compressed") else ""
    print(f"  [{i}] {m.get('ticker','?'):6s}  {m.get('form_type','?'):5s}  "
          f"year={m.get('filing_year','?')}  rerank={score:.3f}  {compressed}")

  [Meta]     {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
  [Rewrite]  "Apple 10-K 2024 net revenue fiscal year total."
  [Queries]  ['"Apple 10-K 2024 net revenue fiscal year total."', 'Apple 2024 annual report net sales total.', 'Apple fiscal year 2024 10-K total revenue disclosure.']
  [Hybrid]   24 BM25 + 24 dense candidates
  [RRF]      28 unique candidates after merge
  [Rerank]   top 5 chunks selected

QUESTION: What was Apple's total net revenue for fiscal year 2024?

ANSWER: Insufficient information in the provided filings to determine Apple's total net revenue for fiscal year 2024.

Metadata extracted : {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
Rewritten query    : "Apple 10-K 2024 net revenue fiscal year total."
Query variants     : ['Apple 2024 annual report net sales total.', 'Apple fiscal year 2024 10-K total revenue disclosure.']

Retrieved 5 chunks:
  [1] AAPL    10-Q   year=2024  rerank=6.763  
  [2] AAPL    10-Q   year=2024  rerank=5.311

In [27]:
demo_q = "What was Apple's total net revenue for fiscal year 2024?"
result = advanced_rag(demo_q, verbose=True)

print()
print("QUESTION:", result["query"])
print()
print("ANSWER:", result["answer"])
print()
print(f"Metadata extracted : {result['metadata']}")
print(f"Rewritten query    : {result['rewritten']}")
print(f"Query variants     : {result['variants']}")
print()
print(f"Retrieved {result['n_chunks']} chunks:")
for i, c in enumerate(result["chunks"], 1):
    m = c["metadata"]
    score = c.get("rerank_score", c.get("score", 0))
    compressed = "(compressed)" if c.get("compressed") else ""
    print(f"  [{i}] {m.get('ticker','?'):6s}  {m.get('form_type','?'):5s}  "
          f"year={m.get('filing_year','?')}  rerank={score:.3f}  {compressed}")

  [Meta]     {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
  [Rewrite]  "Apple 10-K 2024 net revenue fiscal year"
  [Queries]  ['"Apple 10-K 2024 net revenue fiscal year"', 'Apple annual report 2024 net sales', 'Apple 10-K 2024 consolidated statement of income']
  [Hybrid]   24 BM25 + 24 dense candidates
  [RRF]      30 unique candidates after merge
  [Rerank]   top 5 chunks selected

QUESTION: What was Apple's total net revenue for fiscal year 2024?

ANSWER: Based on the provided context, Apple's total net revenue for fiscal year 2024 is $391,035 million [4].

Metadata extracted : {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
Rewritten query    : "Apple 10-K 2024 net revenue fiscal year"
Query variants     : ['Apple annual report 2024 net sales', 'Apple 10-K 2024 consolidated statement of income']

Retrieved 5 chunks:
  [1] AAPL    10-Q   year=2024  rerank=7.301  
  [2] AAPL    10-Q   year=2024  rerank=5.640  
  [3] AAPL    10-K   year=2024  rerank=5.418  
  

In [28]:
demo_q = "What was Apple's total net revenue for fiscal year 2024?"
result = advanced_rag(demo_q, verbose=True)

print()
print("QUESTION:", result["query"])
print()
print("ANSWER:", result["answer"])
print()
print(f"Metadata extracted : {result['metadata']}")
print(f"Rewritten query    : {result['rewritten']}")
print(f"Query variants     : {result['variants']}")
print()
print(f"Retrieved {result['n_chunks']} chunks:")
for i, c in enumerate(result["chunks"], 1):
    m = c["metadata"]
    score = c.get("rerank_score", c.get("score", 0))
    compressed = "(compressed)" if c.get("compressed") else ""
    print(f"  [{i}] {m.get('ticker','?'):6s}  {m.get('form_type','?'):5s}  "
          f"year={m.get('filing_year','?')}  rerank={score:.3f}  {compressed}")

  [Meta]     {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
  [Rewrite]  "Apple 10-K 2024 net revenue fiscal year"
  [Queries]  ['"Apple 10-K 2024 net revenue fiscal year"', 'Apple annual report 2024 net sales fiscal year', 'Apple 10-K 2024 total revenue by fiscal year']
  [Hybrid]   24 BM25 + 24 dense candidates
  [RRF]      23 unique candidates after merge
  [Rerank]   top 5 chunks selected

QUESTION: What was Apple's total net revenue for fiscal year 2024?

ANSWER: To determine Apple's total net revenue for fiscal year 2024, we need to look at the information provided in the 10-K filing [4]. 

According to the filing, Apple's total net sales for fiscal year 2024 were $391,035 million [4].

Metadata extracted : {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
Rewritten query    : "Apple 10-K 2024 net revenue fiscal year"
Query variants     : ['Apple annual report 2024 net sales fiscal year', 'Apple 10-K 2024 total revenue by fiscal year']

Retrieved 5 chunks:
  

In [29]:
demo_q = "What was Apple's total net revenue for fiscal year 2024?"
result = advanced_rag(demo_q, verbose=True)

print()
print("QUESTION:", result["query"])
print()
print("ANSWER:", result["answer"])
print()
print(f"Metadata extracted : {result['metadata']}")
print(f"Rewritten query    : {result['rewritten']}")
print(f"Query variants     : {result['variants']}")
print()
print(f"Retrieved {result['n_chunks']} chunks:")
for i, c in enumerate(result["chunks"], 1):
    m = c["metadata"]
    score = c.get("rerank_score", c.get("score", 0))
    compressed = "(compressed)" if c.get("compressed") else ""
    print(f"  [{i}] {m.get('ticker','?'):6s}  {m.get('form_type','?'):5s}  "
          f"year={m.get('filing_year','?')}  rerank={score:.3f}  {compressed}")

  [Meta]     {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
  [Rewrite]  "Apple 10-K 2024 net revenue fiscal year total."
  [Queries]  ['"Apple 10-K 2024 net revenue fiscal year total."', 'Apple annual report 2024 total revenue for fiscal year.', "What was Apple's total net sales for the fiscal year ended 2024?"]
  [Hybrid]   24 BM25 + 24 dense candidates
  [RRF]      26 unique candidates after merge
  [Rerank]   top 5 chunks selected

QUESTION: What was Apple's total net revenue for fiscal year 2024?

ANSWER: To calculate Apple's total net revenue for fiscal year 2024, we need to sum up the total net sales from the different periods and adjust for deferred revenue.

From [1], the total net sales for the nine months ended June 29, 2024, were $296,105 million. However, this includes $6.5 billion of revenue recognized in the nine months ended June 29, 2024, that was included in deferred revenue as of September 30, 2023. We need to subtract this amount to get the total net sal

## 13. Evaluation (LLM-as-Judge)

Same judge prompts as Baseline RAG for fair comparison.
Results are saved to `./results/advanced_results.csv`.

In [38]:
eval_df = pd.read_csv(CONFIG["eval_path"])
eval_df = eval_df[eval_df["split"] == CONFIG["eval_split"]].reset_index(drop=True)

print(f"Evaluation split : '{CONFIG['eval_split']}'  ({len(eval_df)} questions)")
print(eval_df["question_type"].value_counts().to_string())

Evaluation split : 'dev'  (20 questions)
question_type
extractive     5
paraphrased    5
reasoning      5
adversarial    5


In [39]:
JUDGE_CORRECTNESS_SYSTEM = (
    "Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. "
    "1 = correct value, correct units, correct period. "
    "0 = wrong number, wrong company, wrong period, or completely missing. "
    "Give partial credit for answers close but with unit errors. "
    "Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. "
    "Return a valid JSON object only with keys: "
    "correctness (float 0-1), claims_covered (float 0-1), explanation (string)."
)

JUDGE_FAITHFULNESS_SYSTEM = (
    "Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. "
    "1 = every number and claim is directly supported by the context. "
    "0 = answer contains numbers or claims not present in the context (hallucinated). "
    "Also set claims_covered: fraction of claims in the candidate supported by the context. "
    "Return a valid JSON object only with keys: "
    "faithfulness (float 0-1), claims_covered (float 0-1), explanation (string)."
)


def judge_correctness(question: str, reference: str, candidate: str) -> dict:
    user_msg = (
        f"Question:\n{question}\n\n"
        f"Reference answer:\n{reference}\n\n"
        f"Candidate answer:\n{candidate}"
    )
    raw = call_llm(
        messages=[
            {"role": "system", "content": JUDGE_CORRECTNESS_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        model=CONFIG["judge_model"],
        temperature=CONFIG["temperature_judge"],
        max_tokens=CONFIG["max_tokens_judge"],
        json_mode=True,
    )
    try:
        return json.loads(raw)
    except Exception:
        return {"correctness": 0.0, "claims_covered": 0.0, "explanation": "parse error"}


def judge_faithfulness(question: str, context: str, candidate: str) -> dict:
    user_msg = (
        f"Question:\n{question}\n\n"
        f"Retrieved context:\n{context[:3500]}\n\n"
        f"Candidate answer:\n{candidate}"
    )
    raw = call_llm(
        messages=[
            {"role": "system", "content": JUDGE_FAITHFULNESS_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        model=CONFIG["judge_model"],
        temperature=CONFIG["temperature_judge"],
        max_tokens=CONFIG["max_tokens_judge"],
        json_mode=True,
    )
    try:
        return json.loads(raw)
    except Exception:
        return {"faithfulness": 0.0, "claims_covered": 0.0, "explanation": "parse error"}

In [ ]:
REFUSAL_KEYWORDS = [
    "insufficient", "not contain", "not available", "cannot find",
    "don't have", "no information", "not provided", "unable to",
    "not enough", "not present", "not found",
]

results = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating advanced"):
    question  = row["question"]
    reference = str(row["expected_answer"])

    rag = advanced_rag(question)
    answer  = rag["answer"]
    context = rag["context"]

    corr  = judge_correctness(question, reference, answer)
    faith = judge_faithfulness(question, context, answer)
    refused = any(kw in answer.lower() for kw in REFUSAL_KEYWORDS)

    results.append({
        "question_id":       row["question_id"],
        "question":          question,
        "question_type":     row["question_type"],
        "expected_decision": row.get("expected_decision", "ANSWER"),
        "expected_answer":   reference,
        "predicted_answer":  answer,
        "refused":           refused,
        "correctness":       float(corr.get("correctness", 0.0)),
        "faithfulness":      float(faith.get("faithfulness", 0.0)),
        "claims_covered":    float(corr.get("claims_covered", 0.0)),
        "n_chunks":          rag["n_chunks"],
        "rewritten_query":   rag["rewritten"],
        # "query_variants": json.dumps(rag.get("variants", [])),
        # "context": rag.get("context", ""),
        # "chunks": json.dumps(rag.get("chunks", []), default=str),

        "metadata":          str(rag["metadata"]),
        "corr_explanation":  corr.get("explanation", ""),
        # "faith_explanation": faith.get("explanation", ""),
    })
    

    time.sleep(CONFIG["inter_question_sleep_sec"])

results_df = pd.DataFrame(results)
Path(CONFIG["results_dir"]).mkdir(parents=True, exist_ok=True)
out_path = Path(CONFIG["results_dir"]) / "advanced_results.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved {len(results_df)} rows -> {out_path}")

Evaluating advanced: 100%|██████████| 20/20 [11:32<00:00, 34.63s/it]

Saved 40 rows -> results\advanced_results.csv


In [52]:
results_df 

,question_id,question,question_type,expected_decision,expected_answer,predicted_answer,refused,correctness,faithfulness,claims_covered,n_chunks,rewritten_query,metadata,corr_explanation
0,1.0,What were Apple’s total net sales in fiscal ye...,extractive,ANSWER,Apple's total net sales in fiscal year 2023 we...,"According to the provided context, Apple's tot...",False,0.9,1.00,0.8,5,"""Apple 10-K 2023 net sales fiscal year""","{'ticker': 'AAPL', 'filing_year': 2023, 'form_...","The candidate answer is mostly correct, but th..."
1,2.0,"According to Apple’s fiscal Q3 2025 filing, wh...",extractive,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...",Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Apple 10-Q Q3 2025: countries subject to new ...","{'ticker': 'AAPL', 'filing_year': 2025, 'form_...",The candidate answer is incorrect because it m...
2,3.0,What are the contingencies declared by Microso...,extractive,ANSWER,Microsoft declared a contingency related to IR...,Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Contingencies declared by Microsoft in 10-Q f...","{'ticker': 'MSFT', 'filing_year': 2023, 'form_...",The candidate answer is incorrect as it does n...
3,4.0,Which Microsoft business segment generated the...,extractive,ANSWER,Intelligent Cloud generated the highest revenu...,"Based on the provided context, the Microsoft b...",False,1.0,0.90,1.0,5,"""Revenue by business segment for Microsoft in ...","{'ticker': 'MSFT', 'filing_year': 2023, 'form_...",The candidate answer correctly identifies the ...
4,5.0,What risk does Nvidia mention that could affec...,extractive,ANSWER,Customers may delay adopting new architectures...,"Based on the provided context, the risk that c...",False,0.0,0.75,0.0,5,"""Extract mentions of risks affecting Data Cent...","{'ticker': 'NVDA', 'filing_year': 2025, 'form_...",The candidate answer mentions a shortage of da...
5,26.0,What amount of revenue did Apple generate duri...,paraphrased,ANSWER,Apple's total net sales in fiscal year 2023 we...,Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Revenue net of allowances for fiscal year 202...","{'ticker': 'AAPL', 'filing_year': 2023, 'form_...",The candidate answer is incorrect as it does n...
6,27.0,Which regions were impacted by the newly annou...,paraphrased,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...",Insufficient information in the provided filin...,True,0.0,0.80,0.0,5,"""Region-specific impact of U.S. import duties ...","{'ticker': 'AAPL', 'filing_year': 2025, 'form_...",The candidate answer is incorrect as it does n...
7,28.0,What potential financial obligations did Micro...,paraphrased,ANSWER,Microsoft declared a contingency related to IR...,"Based on the provided context, Microsoft flagg...",True,0.0,0.50,0.0,5,"""Identify Microsoft's 10-Q filings for Q3 2023...","{'ticker': 'MSFT', 'filing_year': 2023, 'form_...",The candidate answer does not mention the corr...
8,29.0,Which of Microsoft’s operating divisions contr...,paraphrased,ANSWER,Intelligent Cloud generated the highest revenu...,Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Revenue breakdown by operating segment for Mi...","{'ticker': 'MSFT', 'filing_year': None, 'form_...",The candidate answer is incorrect as it does n...
9,30.0,What factor does Nvidia identify that could de...,paraphrased,ANSWER,Customers may delay adopting new architectures...,Insufficient information in the provided filings.,True,0.0,1.00,0.0,5,"""Identify factors delaying Data Center revenue...","{'ticker': 'NVDA', 'filing_year': 2025, 'form_...",The candidate answer does not address the ques...


### debug without llm as a judge: output seems more stable?
maybe api call degradation idk

In [32]:
REFUSAL_KEYWORDS = [
    "insufficient", "not contain", "not available", "cannot find",
    "don't have", "no information", "not provided", "unable to",
    "not enough", "not present", "not found",
]

results = []

for _, row in tqdm(eval_df.iloc[:3].iterrows(), total=len(eval_df), desc="Evaluating advanced"):
    question  = row["question"]
    reference = str(row["expected_answer"])

    rag = advanced_rag(question)
    answer  = rag["answer"]
    context = rag["context"]

    # corr  = judge_correctness(question, reference, answer)
    # faith = judge_faithfulness(question, context, answer)
    refused = any(kw in answer.lower() for kw in REFUSAL_KEYWORDS)

    results.append({
        "question_id":       row["question_id"],
        "question":          question,
        "question_type":     row["question_type"],
        "expected_decision": row.get("expected_decision", "ANSWER"),
        "expected_answer":   reference,
        "predicted_answer":  answer,
        "refused":           refused,
        # "correctness":       float(corr.get("correctness", 0.0)),
        # "faithfulness":      float(faith.get("faithfulness", 0.0)),
        # "claims_covered":    float(corr.get("claims_covered", 0.0)),
        "n_chunks":          rag["n_chunks"],
        "rewritten_query":   rag["rewritten"],
        "metadata":          str(rag["metadata"]),
        # "corr_explanation":  corr.get("explanation", ""),
    })
    time.sleep(CONFIG["inter_question_sleep_sec"])

results_df2 = pd.DataFrame(results)
Path(CONFIG["results_dir"]).mkdir(parents=True, exist_ok=True)
out_path = Path(CONFIG["results_dir"]) / "advanced_results.csv"
# results_df.to_csv(out_path, index=False)
print(f"Saved {len(results_df)} rows -> {out_path}")

Evaluating advanced:  15%|█▌        | 3/20 [00:17<01:36,  5.69s/it]

Saved 20 rows -> results\advanced_results.csv


In [33]:
results_df2

,question_id,question,question_type,expected_decision,expected_answer,predicted_answer,refused,n_chunks,rewritten_query,metadata
0,1.0,What were Apple’s total net sales in fiscal ye...,extractive,ANSWER,Apple's total net sales in fiscal year 2023 we...,"According to the provided context, Apple's tot...",False,5,"""Apple 10-K 2023 net sales fiscal year""","{'ticker': 'AAPL', 'filing_year': 2023, 'form_..."
1,2.0,"According to Apple’s fiscal Q3 2025 filing, wh...",extractive,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...","According to Apple's fiscal Q3 2025 filing, th...",False,5,"""Country-specific U.S. import tariffs Q3 2025 ...","{'ticker': 'AAPL', 'filing_year': 2025, 'form_..."
2,3.0,What are the contingencies declared by Microso...,extractive,ANSWER,Microsoft declared a contingency related to IR...,Insufficient information in the provided filin...,True,5,"""Contingencies declared by Microsoft in 10-Q f...","{'ticker': 'MSFT', 'filing_year': 2023, 'form_..."


In [34]:
for i, row in results_df2.iterrows():
    print(f"Row {i}")
    print("Question:", row["question"])
    print("Expected Answer:", row["expected_answer"])
    print("Predicted Answer:", row["predicted_answer"])
    print("Rewritten Query:", row["rewritten_query"])
    print("Metadata:", row["metadata"])
    # print("Correction Explanation:", row["corr_explanation"])
    print("-" * 80)

Row 0
Question: What were Apple’s total net sales in fiscal year 2023?
Expected Answer: Apple's total net sales in fiscal year 2023 were $383,285 million.
Predicted Answer: According to the provided context, Apple's total net sales in fiscal year 2023 were $383.3 billion [1].
Rewritten Query: "Apple 10-K 2023 net sales fiscal year"
Metadata: {'ticker': 'AAPL', 'filing_year': 2023, 'form_type': None}
--------------------------------------------------------------------------------
Row 1
Question: According to Apple’s fiscal Q3 2025 filing, which countries were subject to new U.S. import tariffs?
Expected Answer: According to Apple's Q3 2025 filing, the U.S. imposed additional tariffs on imports from China, India, Japan, South Korea, Taiwan, Vietnam, and the European Union.
Predicted Answer: According to Apple's fiscal Q3 2025 filing, the countries subject to new U.S. import tariffs are:

1. China [1, 2, 3, 5]
2. India [1, 3, 5]
3. Japan [1, 3, 5]
4. South Korea [1, 3, 5]
5. Taiwan [1, 3,

In [35]:
REFUSAL_KEYWORDS = [
    "insufficient", "not contain", "not available", "cannot find",
    "don't have", "no information", "not provided", "unable to",
    "not enough", "not present", "not found",
]

results = []

for _, row in tqdm(eval_df.iloc[:3].iterrows(), total=len(eval_df), desc="Evaluating advanced"):
    question  = row["question"]
    reference = str(row["expected_answer"])

    rag = advanced_rag(question)
    answer  = rag["answer"]
    context = rag["context"]

    corr  = judge_correctness(question, reference, answer)
    faith = judge_faithfulness(question, context, answer)
    refused = any(kw in answer.lower() for kw in REFUSAL_KEYWORDS)

    results.append({
        "question_id":       row["question_id"],
        "question":          question,
        "question_type":     row["question_type"],
        "expected_decision": row.get("expected_decision", "ANSWER"),
        "expected_answer":   reference,
        "predicted_answer":  answer,
        "refused":           refused,
        "correctness":       float(corr.get("correctness", 0.0)),
        "faithfulness":      float(faith.get("faithfulness", 0.0)),
        "claims_covered":    float(corr.get("claims_covered", 0.0)),
        "n_chunks":          rag["n_chunks"],
        "rewritten_query":   rag["rewritten"],
        "metadata":          str(rag["metadata"]),
        "corr_explanation":  corr.get("explanation", ""),
    })
    time.sleep(CONFIG["inter_question_sleep_sec"])

results_df3 = pd.DataFrame(results)
Path(CONFIG["results_dir"]).mkdir(parents=True, exist_ok=True)
out_path = Path(CONFIG["results_dir"]) / "advanced_results.csv"
# results_df.to_csv(out_path, index=False)
print(f"Saved {len(results_df)} rows -> {out_path}")

Evaluating advanced:  15%|█▌        | 3/20 [00:59<05:37, 19.87s/it]

Saved 20 rows -> results\advanced_results.csv


In [37]:
for i, row in results_df3.iterrows():
    print(f"Row {i}")
    print("Question:", row["question"])
    print("Expected Answer:", row["expected_answer"])
    print("Predicted Answer:", row["predicted_answer"])
    print("Rewritten Query:", row["rewritten_query"])
    print("Metadata:", row["metadata"])
    print("Correction Explanation:", row["corr_explanation"])
    print("-" * 80)

Row 0
Question: What were Apple’s total net sales in fiscal year 2023?
Expected Answer: Apple's total net sales in fiscal year 2023 were $383,285 million.
Predicted Answer: According to the provided context, Apple's total net sales in fiscal year 2023 were $383.3 billion [1].
Rewritten Query: "Apple 10-K 2023 net sales fiscal year"
Metadata: {'ticker': 'AAPL', 'filing_year': 2023, 'form_type': None}
Correction Explanation: The candidate answer is mostly correct, but the unit is incorrect. The reference answer is in millions, while the candidate answer is in billions. The number is correct, and the period is correct.
--------------------------------------------------------------------------------
Row 1
Question: According to Apple’s fiscal Q3 2025 filing, which countries were subject to new U.S. import tariffs?
Expected Answer: According to Apple's Q3 2025 filing, the U.S. imposed additional tariffs on imports from China, India, Japan, South Korea, Taiwan, Vietnam, and the European Unio

## 14. Results & Comparison with Baseline

In [54]:
results_df

,question_id,question,question_type,expected_decision,expected_answer,predicted_answer,refused,correctness,faithfulness,claims_covered,n_chunks,rewritten_query,metadata,corr_explanation
0,1.0,What were Apple’s total net sales in fiscal ye...,extractive,ANSWER,Apple's total net sales in fiscal year 2023 we...,"According to the provided context, Apple's tot...",False,0.9,1.00,0.8,5,"""Apple 10-K 2023 net sales fiscal year""","{'ticker': 'AAPL', 'filing_year': 2023, 'form_...","The candidate answer is mostly correct, but th..."
1,2.0,"According to Apple’s fiscal Q3 2025 filing, wh...",extractive,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...",Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Apple 10-Q Q3 2025: countries subject to new ...","{'ticker': 'AAPL', 'filing_year': 2025, 'form_...",The candidate answer is incorrect because it m...
2,3.0,What are the contingencies declared by Microso...,extractive,ANSWER,Microsoft declared a contingency related to IR...,Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Contingencies declared by Microsoft in 10-Q f...","{'ticker': 'MSFT', 'filing_year': 2023, 'form_...",The candidate answer is incorrect as it does n...
3,4.0,Which Microsoft business segment generated the...,extractive,ANSWER,Intelligent Cloud generated the highest revenu...,"Based on the provided context, the Microsoft b...",False,1.0,0.90,1.0,5,"""Revenue by business segment for Microsoft in ...","{'ticker': 'MSFT', 'filing_year': 2023, 'form_...",The candidate answer correctly identifies the ...
4,5.0,What risk does Nvidia mention that could affec...,extractive,ANSWER,Customers may delay adopting new architectures...,"Based on the provided context, the risk that c...",False,0.0,0.75,0.0,5,"""Extract mentions of risks affecting Data Cent...","{'ticker': 'NVDA', 'filing_year': 2025, 'form_...",The candidate answer mentions a shortage of da...
5,26.0,What amount of revenue did Apple generate duri...,paraphrased,ANSWER,Apple's total net sales in fiscal year 2023 we...,Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Revenue net of allowances for fiscal year 202...","{'ticker': 'AAPL', 'filing_year': 2023, 'form_...",The candidate answer is incorrect as it does n...
6,27.0,Which regions were impacted by the newly annou...,paraphrased,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...",Insufficient information in the provided filin...,True,0.0,0.80,0.0,5,"""Region-specific impact of U.S. import duties ...","{'ticker': 'AAPL', 'filing_year': 2025, 'form_...",The candidate answer is incorrect as it does n...
7,28.0,What potential financial obligations did Micro...,paraphrased,ANSWER,Microsoft declared a contingency related to IR...,"Based on the provided context, Microsoft flagg...",True,0.0,0.50,0.0,5,"""Identify Microsoft's 10-Q filings for Q3 2023...","{'ticker': 'MSFT', 'filing_year': 2023, 'form_...",The candidate answer does not mention the corr...
8,29.0,Which of Microsoft’s operating divisions contr...,paraphrased,ANSWER,Intelligent Cloud generated the highest revenu...,Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Revenue breakdown by operating segment for Mi...","{'ticker': 'MSFT', 'filing_year': None, 'form_...",The candidate answer is incorrect as it does n...
9,30.0,What factor does Nvidia identify that could de...,paraphrased,ANSWER,Customers may delay adopting new architectures...,Insufficient information in the provided filings.,True,0.0,1.00,0.0,5,"""Identify factors delaying Data Center revenue...","{'ticker': 'NVDA', 'filing_year': 2025, 'form_...",The candidate answer does not address the ques...


In [55]:
print("=" * 60)
print(" ADVANCED RAG -- EVALUATION RESULTS")
print("=" * 60)

print(f"\nOverall (n={len(results_df)}):")
for col in ["correctness", "faithfulness", "claims_covered"]:
    print(f"  {col:20s}: {results_df[col].mean():.3f}")

print("\nBy question type:")
by_type = (
    results_df
    .groupby("question_type")[["correctness", "faithfulness", "claims_covered"]]
    .mean()
    .round(3)
)
print(by_type.to_string())

adv = results_df[
    (results_df["question_type"] == "adversarial") &
    (results_df["expected_decision"] == "REFUSE")
]
if len(adv):
    print(f"\nAdversarial refusal accuracy: {adv['refused'].mean():.2%}  ({adv['refused'].sum()}/{len(adv)})")

# Compare with baseline if available
baseline_path = Path(CONFIG["results_dir"]) / "baseline_results.csv"
if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path)
    print("\n" + "=" * 60)
    print(" COMPARISON: BASELINE vs ADVANCED RAG")
    print("=" * 60)
    comparison = pd.DataFrame({
        "Baseline": baseline_df[["correctness", "faithfulness", "claims_covered"]].mean(),
        "Advanced": results_df[["correctness", "faithfulness", "claims_covered"]].mean(),
    }).round(3)
    comparison["Delta"] = (comparison["Advanced"] - comparison["Baseline"]).round(3)
    comparison["Improvement"] = comparison["Delta"].apply(
        lambda x: f"+{x:.3f}" if x >= 0 else f"{x:.3f}"
    )
    print(comparison.to_string())
else:
    print("\n(Run baseline_rag.ipynb first to see comparison)")

 ADVANCED RAG -- EVALUATION RESULTS

Overall (n=20):
  correctness         : 0.475
  faithfulness        : 0.878
  claims_covered      : 0.270

By question type:
               correctness  faithfulness  claims_covered
question_type                                           
adversarial           0.80          1.00            0.00
extractive            0.38          0.93            0.36
paraphrased           0.00          0.86            0.00
reasoning             0.72          0.72            0.72

Adversarial refusal accuracy: 100.00%  (5/5)

 COMPARISON: BASELINE vs ADVANCED RAG
                Baseline  Advanced  Delta Improvement
correctness        0.225     0.475  0.250      +0.250
faithfulness       0.980     0.878 -0.102      -0.102
claims_covered     0.000     0.270  0.270      +0.270


In [42]:
results_df.iloc[2].question

'According to Apple’s fiscal Q3 2025 filing, which countries were subject to new U.S. import tariffs?'

In [56]:
results_df

,question_id,question,question_type,expected_decision,expected_answer,predicted_answer,refused,correctness,faithfulness,claims_covered,n_chunks,rewritten_query,metadata,corr_explanation
0,1.0,What were Apple’s total net sales in fiscal ye...,extractive,ANSWER,Apple's total net sales in fiscal year 2023 we...,"According to the provided context, Apple's tot...",False,0.9,1.00,0.8,5,"""Apple 10-K 2023 net sales fiscal year""","{'ticker': 'AAPL', 'filing_year': 2023, 'form_...","The candidate answer is mostly correct, but th..."
1,2.0,"According to Apple’s fiscal Q3 2025 filing, wh...",extractive,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...",Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Apple 10-Q Q3 2025: countries subject to new ...","{'ticker': 'AAPL', 'filing_year': 2025, 'form_...",The candidate answer is incorrect because it m...
2,3.0,What are the contingencies declared by Microso...,extractive,ANSWER,Microsoft declared a contingency related to IR...,Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Contingencies declared by Microsoft in 10-Q f...","{'ticker': 'MSFT', 'filing_year': 2023, 'form_...",The candidate answer is incorrect as it does n...
3,4.0,Which Microsoft business segment generated the...,extractive,ANSWER,Intelligent Cloud generated the highest revenu...,"Based on the provided context, the Microsoft b...",False,1.0,0.90,1.0,5,"""Revenue by business segment for Microsoft in ...","{'ticker': 'MSFT', 'filing_year': 2023, 'form_...",The candidate answer correctly identifies the ...
4,5.0,What risk does Nvidia mention that could affec...,extractive,ANSWER,Customers may delay adopting new architectures...,"Based on the provided context, the risk that c...",False,0.0,0.75,0.0,5,"""Extract mentions of risks affecting Data Cent...","{'ticker': 'NVDA', 'filing_year': 2025, 'form_...",The candidate answer mentions a shortage of da...
5,26.0,What amount of revenue did Apple generate duri...,paraphrased,ANSWER,Apple's total net sales in fiscal year 2023 we...,Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Revenue net of allowances for fiscal year 202...","{'ticker': 'AAPL', 'filing_year': 2023, 'form_...",The candidate answer is incorrect as it does n...
6,27.0,Which regions were impacted by the newly annou...,paraphrased,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...",Insufficient information in the provided filin...,True,0.0,0.80,0.0,5,"""Region-specific impact of U.S. import duties ...","{'ticker': 'AAPL', 'filing_year': 2025, 'form_...",The candidate answer is incorrect as it does n...
7,28.0,What potential financial obligations did Micro...,paraphrased,ANSWER,Microsoft declared a contingency related to IR...,"Based on the provided context, Microsoft flagg...",True,0.0,0.50,0.0,5,"""Identify Microsoft's 10-Q filings for Q3 2023...","{'ticker': 'MSFT', 'filing_year': 2023, 'form_...",The candidate answer does not mention the corr...
8,29.0,Which of Microsoft’s operating divisions contr...,paraphrased,ANSWER,Intelligent Cloud generated the highest revenu...,Insufficient information in the provided filin...,True,0.0,1.00,0.0,5,"""Revenue breakdown by operating segment for Mi...","{'ticker': 'MSFT', 'filing_year': None, 'form_...",The candidate answer is incorrect as it does n...
9,30.0,What factor does Nvidia identify that could de...,paraphrased,ANSWER,Customers may delay adopting new architectures...,Insufficient information in the provided filings.,True,0.0,1.00,0.0,5,"""Identify factors delaying Data Center revenue...","{'ticker': 'NVDA', 'filing_year': 2025, 'form_...",The candidate answer does not address the ques...


In [57]:
for i, row in results_df.iterrows():
    print(f"Row {i}")
    print("Question:", row["question"])
    print("Expected Answer:", row["expected_answer"])
    print("Predicted Answer:", row["predicted_answer"])
    print("Rewritten Query:", row["rewritten_query"])
    print("Metadata:", row["metadata"])
    print("Correction Explanation:", row["corr_explanation"])
    print("-" * 80)

Row 0
Question: What were Apple’s total net sales in fiscal year 2023?
Expected Answer: Apple's total net sales in fiscal year 2023 were $383,285 million.
Predicted Answer: According to the provided context, Apple's total net sales in fiscal year 2023 were $383.3 billion [1].
Rewritten Query: "Apple 10-K 2023 net sales fiscal year"
Metadata: {'ticker': 'AAPL', 'filing_year': 2023, 'form_type': None}
Correction Explanation: The candidate answer is mostly correct, but the unit is incorrect. The reference answer is in millions, while the candidate answer is in billions. The number is correct, and the period is correct.
--------------------------------------------------------------------------------
Row 1
Question: According to Apple’s fiscal Q3 2025 filing, which countries were subject to new U.S. import tariffs?
Expected Answer: According to Apple's Q3 2025 filing, the U.S. imposed additional tariffs on imports from China, India, Japan, South Korea, Taiwan, Vietnam, and the European Unio